## Setup

**Test date: 22.4.2026**

**Claude Sonnet 4.6** was tasked with generating a **chair** using the **SPOMSO** package. Claude was provided with the **.zip** file of the **Aegolius** GitHub repository (release SPOMSO 1.3.0) and the following prompt:

*Read through this zip file. It contains the repo of the Python package SPOMSO. I want you to design a chair using this Python package. As the output I want some code that I would run and it would output the point cloud of a chair and some figures. Everything else I leave up to you. Good luck and if you have any questions feel free to ask.*

## Response:

Now I have a thorough understanding of the library. Let me install it and design the chair.Now I have everything I need. Let me write the chair design script:All outputs generated successfully. Let me copy everything to the output directory:---

Here's what the script does and produces:

**Chair design** — a modern 4-legged chair built entirely from SPOMSO SDFs:
- **Seat** — a rounded flat `Box`
- **Back rest** — a tall rounded `Box`, tilted 5° backward, with a top rail
- **4 legs** — `Cylinder`s at the four corners with slight rounding
- **Apron rails** — 4 thin `Box`es connecting the legs beneath the seat for rigidity

All parts are combined using SPOMSO's `CombineGeometry("UNION")` into a single SDF, then evaluated on an 80×80×120 voxel grid.

**Outputs:**
| File | Description |
|---|---|
| `chair_spomso.py` | The self-contained script to run |
| `chair_point_cloud.npy` | ~72 k surface points as a numpy array, shape `(N, 3)` |
| `chair_side_xz.png` | XZ cross-section (side view) |
| `chair_front_yz.png` | YZ cross-section (front view) |
| `chair_pointcloud_3d.png` | 3D scatter of the surface cloud, coloured by height |
| `chair_interactive.html` | Plotly interactive volume render (open in a browser) |

To run it yourself, just install the package (`pip install -e Aegolius-main/Code/spomso`) and then `python chair_spomso.py`. The resolution is easily tunable — bump `CO_RESOLUTION` up to `(150, 150, 220)` for a denser point cloud.

In [ ]:
"""
Chair design using the SPOMSO package.

This script constructs a modern four-legged chair using Signed Distance
Functions (SDFs).  It then:
  - saves the surface point cloud as a .npy file
  - produces three matplotlib figures
      1. XZ cross-section (side view)
      2. YZ cross-section (front view)
      3. 3-D scatter plot of the surface point cloud coloured by height
  - shows an interactive Plotly volume render

Run:
    python chair_spomso.py
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from time import process_time

from spomso.cores.helper_functions import generate_grid, smarter_reshape
from spomso.cores.post_processing import hard_binarization
from spomso.cores.geom_3d import Box, Cylinder
from spomso.cores.combine import CombineGeometry

# ──────────────────────────────────────────────────────────────────────────────
# GRID  (units are in arbitrary "design units" – think of 1 unit ≈ 50 cm)
# The chair fits in a 1.2 × 1.2 × 1.8 bounding box.
# ──────────────────────────────────────────────────────────────────────────────
CO_SIZE       = (1.2, 1.2, 1.8)   # (x-width, y-depth, z-height)
CO_RESOLUTION = (80, 80, 120)      # voxel grid – increase for more detail

coor, co_res_new = generate_grid(CO_SIZE, CO_RESOLUTION)

# ──────────────────────────────────────────────────────────────────────────────
# CHAIR GEOMETRY
#
# Coordinate conventions
#   x  →  left / right
#   y  →  front / back
#   z  ↑  up / down
#   origin at the centre of the grid
#
# Chair layout (z is measured from grid centre = 0):
#   legs:        z  ∈ [-0.85 … -0.10]   length 0.75
#   seat:        z  ∈ [-0.10 …  0.00]   thickness 0.10
#   back-rest:   z  ∈ [ 0.00 …  0.65]   above the seat
#
# Four legs placed at (±0.42, ±0.42).
# ──────────────────────────────────────────────────────────────────────────────

union = CombineGeometry("UNION")

# ------------------------------------------------------------------
# 1. SEAT  –  a rounded flat box
# ------------------------------------------------------------------
seat = Box(0.90, 0.90, 0.10)          # wide, deep, thin
seat.rounding(0.04)
seat.move((0.0, 0.0, -0.05))          # top surface sits at z = 0

# ------------------------------------------------------------------
# 2. BACK REST  –  vertical slab behind the seat, slightly tapered
#    We build it as a rounded box and tilt it 5° backward.
# ------------------------------------------------------------------
backrest = Box(0.84, 0.06, 0.65)
backrest.rounding(0.04)
# position: centred in x, rear edge of seat in y, rising from seat top
backrest.move((0.0, -0.42, 0.325))    # z=0 → z=0.65
# tilt backward ~5°
backrest.rotate(-np.deg2rad(5), (1, 0, 0))

# ── horizontal slat across the top of the back rest ──────────────
top_rail = Box(0.84, 0.08, 0.06)
top_rail.rounding(0.03)
top_rail.move((0.0, -0.42, 0.62))
top_rail.rotate(-np.deg2rad(5), (1, 0, 0))

# ------------------------------------------------------------------
# 3. LEGS  –  four cylinders, one at each corner
#    Leg centre-to-centre from seat edge inset by 0.03
# ------------------------------------------------------------------
leg_radius  = 0.045
leg_height  = 0.75
leg_offset  = 0.42                    # how far from centre in x and y
leg_z_centre = -0.85 + leg_height/2  # = -0.475  (bottom at -0.85)

leg_positions = [
    ( leg_offset,  leg_offset, leg_z_centre),
    ( leg_offset, -leg_offset, leg_z_centre),
    (-leg_offset,  leg_offset, leg_z_centre),
    (-leg_offset, -leg_offset, leg_z_centre),
]

legs = []
for pos in leg_positions:
    leg = Cylinder(leg_radius, leg_height)
    leg.rounding(0.01)
    leg.move(pos)
    legs.append(leg)

# ------------------------------------------------------------------
# 4. SEAT SUPPORT RAILS  –  thin horizontal boxes connecting the legs
#    under the seat for rigidity (like an apron)
# ------------------------------------------------------------------
rail_thickness = 0.04
rail_height    = 0.12
rail_z         = -0.16                # just below the seat

# front & back rails (run along x)
front_rail = Box(0.80, rail_thickness, rail_height)
front_rail.rounding(0.01)
front_rail.move((0.0,  leg_offset, rail_z))

back_rail = Box(0.80, rail_thickness, rail_height)
back_rail.rounding(0.01)
back_rail.move((0.0, -leg_offset, rail_z))

# left & right rails (run along y – shorter to fit between leg faces)
side_rail_depth = 0.80
left_rail = Box(rail_thickness, side_rail_depth, rail_height)
left_rail.rounding(0.01)
left_rail.move(( leg_offset, 0.0, rail_z))

right_rail = Box(rail_thickness, side_rail_depth, rail_height)
right_rail.rounding(0.01)
right_rail.move((-leg_offset, 0.0, rail_z))

# ------------------------------------------------------------------
# 5. UNION everything together
# ------------------------------------------------------------------
print("Evaluating SDF …")
t0 = process_time()

chair = union.combine(
    seat,
    backrest,
    top_rail,
    *legs,
    front_rail,
    back_rail,
    left_rail,
    right_rail,
)

chair_sdf = chair.create(coor)

print(f"  done in {process_time()-t0:.2f} s")

# ──────────────────────────────────────────────────────────────────────────────
# BINARISE  (interior = 1, exterior = 0)
# ──────────────────────────────────────────────────────────────────────────────
binary_flat  = hard_binarization(chair_sdf, 0)
binary_3d    = smarter_reshape(binary_flat, CO_RESOLUTION)

# ──────────────────────────────────────────────────────────────────────────────
# SURFACE POINT CLOUD
#   Keep only voxels very close to the zero level-set (|SDF| < 1 voxel step)
# ──────────────────────────────────────────────────────────────────────────────
dx = CO_SIZE[0] / CO_RESOLUTION[0]
surface_mask = np.abs(chair_sdf) < dx * 1.5
surface_pts  = coor[:, surface_mask].T           # shape (N, 3)

print(f"Surface point cloud: {surface_pts.shape[0]:,} points")
np.save("chair_point_cloud.npy", surface_pts)
print("Saved → chair_point_cloud.npy")

# ──────────────────────────────────────────────────────────────────────────────
# FIGURE 1  –  side cross-section  (XZ plane, y = 0)
# ──────────────────────────────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 9))
ax1.set_facecolor("#1a1a2e")
fig1.patch.set_facecolor("#1a1a2e")

ax1.imshow(
    binary_3d[:, CO_RESOLUTION[1] // 2, :].T,
    cmap="Blues",
    origin="lower",
    extent=(-CO_SIZE[0]/2, CO_SIZE[0]/2, -CO_SIZE[2]/2, CO_SIZE[2]/2),
    aspect="equal",
)
ax1.set_xlabel("x (left–right)", color="white")
ax1.set_ylabel("z (up–down)",    color="white")
ax1.set_title("Chair – Side Cross-section (XZ)", color="white", fontsize=14)
ax1.tick_params(colors="white")
for spine in ax1.spines.values():
    spine.set_edgecolor("white")
ax1.grid(color="white", alpha=0.2, linestyle="--")
fig1.tight_layout()
fig1.savefig("chair_side_xz.png", dpi=150, bbox_inches="tight")
print("Saved → chair_side_xz.png")

# ──────────────────────────────────────────────────────────────────────────────
# FIGURE 2  –  front cross-section  (YZ plane, x = 0)
# ──────────────────────────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 9))
ax2.set_facecolor("#1a1a2e")
fig2.patch.set_facecolor("#1a1a2e")

ax2.imshow(
    binary_3d[CO_RESOLUTION[0] // 2, :, :].T,
    cmap="Purples",
    origin="lower",
    extent=(-CO_SIZE[1]/2, CO_SIZE[1]/2, -CO_SIZE[2]/2, CO_SIZE[2]/2),
    aspect="equal",
)
ax2.set_xlabel("y (front–back)", color="white")
ax2.set_ylabel("z (up–down)",    color="white")
ax2.set_title("Chair – Front Cross-section (YZ)", color="white", fontsize=14)
ax2.tick_params(colors="white")
for spine in ax2.spines.values():
    spine.set_edgecolor("white")
ax2.grid(color="white", alpha=0.2, linestyle="--")
fig2.tight_layout()
fig2.savefig("chair_front_yz.png", dpi=150, bbox_inches="tight")
print("Saved → chair_front_yz.png")

# ──────────────────────────────────────────────────────────────────────────────
# FIGURE 3  –  3-D scatter plot of the surface point cloud
# ──────────────────────────────────────────────────────────────────────────────
# Subsample for speed / readability
rng     = np.random.default_rng(42)
n_show  = min(15_000, surface_pts.shape[0])
idx     = rng.choice(surface_pts.shape[0], n_show, replace=False)
pts     = surface_pts[idx]

fig3    = plt.figure(figsize=(10, 10))
fig3.patch.set_facecolor("#1a1a2e")
ax3     = fig3.add_subplot(111, projection="3d")
ax3.set_facecolor("#1a1a2e")

z_vals  = pts[:, 2]
z_norm  = (z_vals - z_vals.min()) / (z_vals.max() - z_vals.min())
colors  = cm.plasma(z_norm)

ax3.scatter(
    pts[:, 0], pts[:, 1], pts[:, 2],
    c=colors, s=0.5, alpha=0.7, linewidths=0,
)
ax3.set_xlabel("x", color="white")
ax3.set_ylabel("y", color="white")
ax3.set_zlabel("z", color="white")
ax3.set_title("Chair – Surface Point Cloud", color="white", fontsize=14, pad=15)
ax3.tick_params(colors="white")
ax3.xaxis.pane.fill = False
ax3.yaxis.pane.fill = False
ax3.zaxis.pane.fill = False
ax3.xaxis.pane.set_edgecolor("gray")
ax3.yaxis.pane.set_edgecolor("gray")
ax3.zaxis.pane.set_edgecolor("gray")
ax3.grid(color="gray", alpha=0.3)

# Colour-bar for height
sm = cm.ScalarMappable(cmap="plasma",
                       norm=plt.Normalize(z_vals.min(), z_vals.max()))
sm.set_array([])
cbar = fig3.colorbar(sm, ax=ax3, shrink=0.6, pad=0.1)
cbar.set_label("Height z", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

fig3.tight_layout()
fig3.savefig("chair_pointcloud_3d.png", dpi=150, bbox_inches="tight")
print("Saved → chair_pointcloud_3d.png")

# ──────────────────────────────────────────────────────────────────────────────
# PLOTLY INTERACTIVE VOLUME RENDER
# ──────────────────────────────────────────────────────────────────────────────
try:
    import plotly.graph_objects as go

    fig_ply = go.Figure(data=go.Volume(
        x=coor[0],
        y=coor[1],
        z=coor[2],
        value=binary_flat,
        isomin=0.5,
        isomax=1.0,
        opacity=0.08,
        surface_count=3,
        colorscale="Blues",
    ))
    fig_ply.update_layout(
        title="Chair – Interactive 3D Volume",
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            bgcolor="#1a1a2e",
        ),
        paper_bgcolor="#1a1a2e",
        font_color="white",
    )
    fig_ply.write_html("chair_interactive.html")
    print("Saved → chair_interactive.html")
    fig_ply.show()
except ImportError:
    print("plotly not available – skipping interactive render.")

plt.show()
print("\nAll done!")

## Outcome

***The entire output is too large to show here, but the code runs successfully and does generate a chair. The proportions of the chair are slightly off and the backrest is tilted the wrong direction but overall the task was successfully completed, even though Claude had no prior knowledge about the SPOMSO package.***

The entire **chair** can be visualized with the code below:

In [ ]:
import plotly.graph_objects as go
fig = go.Figure(data=go.Volume(
    x=coor[0],
    y=coor[1],
    z=coor[2],
    value=binary_flat,
    isomin=0.9,
    isomax=1.0,
    opacity=0.9,
    surface_count=1,
))
fig.show()

Claude was tasked with generating **Markdown (.md) guides** to assist **users** and **LLMs** in utilizing the **SDF**, **vector field**, and **automatic differentiation** functionalities of the **SPOMSO** package. These guides have been refined to correct minor errors and are accessible via the [**guides**](./guides/) directory